# VCO-01 | Cross-coupled LC VCO — reproducible L3/L4 experiment

**课程**：PLL 第 8 章《VCO 与 DCO 可控振荡器》  
**L3（作者发布）**：阅读已执行的完整 Notebook，查看 27 组数据、指标、起振波形和一个完整采样的原始时域文件。  
**L4（读者复现）**：登录 Kaggle → Copy & Edit → 修改下面参数 → Run All；结果保存在自己会话的 `/kaggle/working/`。

**这是 SPICE，不是纯公式。**原始包的运行器是 **Node.js + EEcircuit-engine 1.9.0 + ngspice 45.2 WASM**，晶体管模型为 SKY130 1.8 V NMOS TT。这里复用原作者的 `run.mjs`、`vco.cir`、`models/` 和 `baseline.csv`，**不另写晶体管等效模型**。

**运行准备**：Kaggle Notebook CPU 模式，启用 Internet，允许下载本仓库公开 ZIP、Node.js 和 npm 依赖；无需 GPU。**从未发布/运行的源 Notebook 不能标为 L3 已完成，发布前必须完成真实测试。**

**边界**：输入为固定 LC 电容工作点，没有变容管连续调谐；仿真为瞬态与 FFT，不是振荡器 PSS/PNoise，相应 FFT 曲线不能当作 phase-noise dBc/Hz。


In [ ]:
# [L4 PARAMETERS] Change these values before running your own detailed case.
L_NH = 1.0
C_FF = 810
RS_OHM = 2.2
PAIR_W_UM = 2.4
PAIR_L_UM = 0.16
TAIL_BIAS_V = 0.90
START_KICK_UA = 50

# L3: always reproduce the canonical 27-case L2 grid.
# L4: one detailed case generates raw CSV with every transient sample.
RUN_CANONICAL_GRID = True
RUN_DETAILED_CASE = True
RUN_FULL_RAW_GRID = False  # optional and storage-intensive (27 huge raw CSV files)
SIMULATION_TIMEOUT_S = 7200


In [ ]:
import csv, datetime as dt, hashlib, json, os, platform, shutil
import subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

SOURCE_REF = "dd2c0b29d0045c960b885c2a36c4fc644ab12066"  # pin model/netlist/runner/export code in GitHub
GITHUB_RAW = f"https://raw.githubusercontent.com/thecloud723/analog-ic-learning/{SOURCE_REF}"
WORK = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "vco01_runs") / "VCO-01"
WORK.mkdir(parents=True, exist_ok=True)
PACKAGE_ZIP = WORK / "lc-vco-lab.zip"
LAB = WORK / "lc-vco-lab"
EXPORTER = WORK / "export_l2.py"

def download(url: str, dest: Path):
    req = urllib.request.Request(url, headers={"User-Agent": "analog-ic-learning-vco01/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as response, dest.open("wb") as f:
            shutil.copyfileobj(response, f)
    except Exception as e:
        raise RuntimeError(
            f"Download failed: {url}. On Kaggle enable Internet in notebook settings "
            "or attach the public ZIP and script as a Kaggle dataset."
        ) from e
    print(dest.name, dest.stat().st_size, "bytes")

download(f"{GITHUB_RAW}/pll/assets/lc-vco-lab.zip", PACKAGE_ZIP)
download(f"{GITHUB_RAW}/pll/labs/VCO-01/export_l2.py", EXPORTER)
with zipfile.ZipFile(PACKAGE_ZIP) as archive:
    # Explicitly reject path traversal in public archives.
    for info in archive.infolist():
        target = (WORK / info.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise RuntimeError("unsafe archive member: " + info.filename)
    archive.extractall(WORK)
assert (LAB / "run.mjs").exists() and (LAB / "vco.cir").exists()
assert (LAB / "models/sky130_fd_pr__nfet_01v8__tt.pm3.spice").exists()
print("ZIP SHA256:", hashlib.sha256(PACKAGE_ZIP.read_bytes()).hexdigest())
print("GitHub source commit:", SOURCE_REF)
print("Circuit + SKY130 model assets verified")


## Environment setup

The source ZIP requires **Node.js ≥22**. If this CPU instance does not have Node 22, create a private portable Node environment using `nodeenv` (no system-wide admin change). Then install the **locked** npm dependencies using `npm ci`.

If Kaggle cannot access npm or the Node download endpoint, the Notebook cannot be marked L4 verified; attach the missing dependencies as a versioned Kaggle dataset and retest. No secrets/tokens are necessary for the **public** source ZIP.

In [ ]:
def node_version(command):
    try:
        output = subprocess.check_output([str(command), "--version"], text=True).strip()
        return int(output.lstrip("v").split(".")[0]), output
    except Exception:
        return 0, "unavailable"

NODE = shutil.which("node")
major, version = node_version(NODE) if NODE else (0, "not installed")
if major < 22:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nodeenv"], check=True, timeout=600)
    NODE_ENV = WORK / "nodeenv"
    if not (NODE_ENV / "bin/node").exists():
        subprocess.run(["nodeenv", "--node=22.16.0", "--prebuilt", str(NODE_ENV)],
                       check=True, timeout=900)
    NODE = str(NODE_ENV / "bin/node")
    major, version = node_version(NODE)
assert major >= 22, "Node.js >=22 is required for the original run.mjs"
ENV = os.environ.copy()
ENV["PATH"] = str(Path(NODE).parent) + os.pathsep + ENV.get("PATH", "")
print("Node", version)
subprocess.run(["npm", "ci", "--no-audit", "--no-fund"],
               cwd=LAB, env=ENV, check=True, timeout=900)
print("npm", subprocess.check_output(["npm", "--version"], env=ENV, text=True).strip())
print("Pinned simulation engine: EEcircuit-engine 1.9.0 / ngspice 45.2 WASM")


## Canonical 27-case sweep (author L3 → tutorial L2)

Axes: L = 0.8 / 1.0 / 1.2 nH; C = 680 / 810 / 970 fF; Rs = 2.2 / 3.3 / 18 Ω. All other parameters remain the source defaults. `--web` writes **27 concise but physically simulated transient traces and metrics**; this matches the existing HTML interactive dataset design.

For an L3 publication, `RUN_CANONICAL_GRID` must be true. If the job fails or baseline validation fails, do not publish a success claim.

In [ ]:
def execute(label, args, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    command = [str(NODE), "run.mjs", *args, "--out", str(out_dir)]
    print("RUN:", " ".join(command), flush=True)
    start = time.monotonic()
    result = subprocess.run(command, cwd=LAB, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            timeout=SIMULATION_TIMEOUT_S)
    elapsed = time.monotonic() - start
    log = WORK / f"{label}.log"
    log.write_text(result.stdout, encoding="utf-8")
    print(f"{label}: returncode={result.returncode}, elapsed={elapsed:.1f}s")
    print("LOG:", log)
    print("\n".join(result.stdout.splitlines()[-8:]))
    if result.returncode:
        raise RuntimeError(f"{label} failed; examine {log}")
    return elapsed

if RUN_CANONICAL_GRID:
    web_dir = WORK / "web-grid"
    grid_seconds = execute("grid_27", ["--web"], web_dir)
    grid = json.loads((web_dir / "web-data.json").read_text(encoding="utf-8"))
    assert len(grid) == 27, f"Unexpected grid size: {len(grid)}"
    print("27 SPICE cases finished. State counts:",
          {state: sum(x["state"] == state for x in grid)
           for state in sorted({x["state"] for x in grid})})
else:
    print("Canonical grid skipped; L2 cannot be exported or L3 fully validated.")


In [ ]:
if RUN_CANONICAL_GRID:
    l2_dir = WORK / "L2-export"
    subprocess.run(
        [sys.executable, str(EXPORTER),
         "--web-json", str(web_dir / "web-data.json"),
         "--baseline", str(LAB / "baseline.csv"),
         "--package-zip", str(PACKAGE_ZIP),
         "--source-commit", SOURCE_REF,
         "--output-dir", str(l2_dir)],
        check=True, timeout=120)
    manifest = json.loads((l2_dir / "l2_manifest.json").read_text(encoding="utf-8"))
    assert manifest["cases"] == 27
    print("L2 verified:", l2_dir)
    print("L2 values are unmodified selections from the same L3 run; no JS approximation.")


## One detailed transistor-level transient (L4 parameter editing)

Unlike `--web`, the normal run produces the **full time-sampled CSV** `case-01.csv`, envelope SVG and metrics. Modify the first cell (especially L, C and Rs) to explore start-up success/decay, MOS width and bias. The standard record length is 120 ns with a maximum timestep of 2 ps.

The one-shot start kick means that the reported “steady” decision is conditional on the observation window (80–120 ns); it is not a proof of long-term stability.

In [ ]:
if RUN_DETAILED_CASE:
    detail_args = [
        "--l", str(L_NH), "--c", str(C_FF), "--rs", str(RS_OHM),
        "--w", str(PAIR_W_UM), "--length", str(PAIR_L_UM),
        "--bias", str(TAIL_BIAS_V), "--kick", str(START_KICK_UA)]
    detail_dir = WORK / "detailed-case"
    detail_seconds = execute("detail", detail_args, detail_dir)
    import pandas as pd
    summary = pd.read_csv(detail_dir / "metrics.csv")
    display(summary)
    raw = pd.read_csv(detail_dir / "case-01.csv")
    print("Full waveform rows:", len(raw), "columns:", list(raw.columns))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
FIGURES = WORK / "figures"
FIGURES.mkdir(exist_ok=True)
if RUN_CANONICAL_GRID:
    focus = [x for x in grid if float(x["l"]) == 1.0 and float(x["c"]) == 810.0]
    fig, ax = plt.subplots(figsize=(9, 4))
    for row in sorted(focus, key=lambda x: x["rs"]):
        ax.plot(np.arange(len(row["env"])) * 0.5, row["env"],
                label=f'Rs={row["rs"]} Ohm ({row["state"]})')
    ax.set(xlabel="Time (ns)", ylabel="Max abs differential voltage per 0.5 ns (V)",
           title="SKY130 LC VCO startup vs tank loss: L=1nH, C=810fF")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURES / "startup_envelope.png", dpi=150)
    plt.show()

if RUN_DETAILED_CASE:
    fig, ax = plt.subplots(figsize=(9, 4))
    view = raw[(raw.time_ns >= 100) & (raw.time_ns <= 102)]
    ax.plot(view.time_ns, view.outp_v, label="V(outp)")
    ax.plot(view.time_ns, view.outn_v, label="V(outn)")
    ax.set(xlabel="Time (ns)", ylabel="Voltage (V)", title="Detailed SPICE steady-window waveform")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURES / "steady_waveform.png", dpi=150)
    plt.show()


## Optional FFT: transient spectrum, **not oscillator phase noise**

The following plot is a finite-window FFT of one resampled transient waveform. Spectral peaks are in **dBc relative to the largest non-DC bin**; binwidth, windowing and leakage all matter. It is not a PSS/PNoise result and should **not** be labeled phase-noise dBc/Hz.

In [ ]:
if RUN_DETAILED_CASE:
    window = raw[(raw.time_ns >= 100) & (raw.time_ns <= 120)]
    assert len(window) > 100, "Not enough time samples for FFT"
    t_ns = np.arange(100.0, 120.0, 0.01)  # 10 ps interpolation spacing
    differential = np.interp(t_ns, window.time_ns,
                             window.outp_v - window.outn_v)
    differential -= differential.mean()
    hann = np.hanning(len(differential))
    spectrum = np.abs(np.fft.rfft(differential * hann))
    f_ghz = np.fft.rfftfreq(len(differential), d=0.01e-9) / 1e9
    spectrum[0] = 0
    ref = max(float(spectrum.max()), 1e-30)
    db_rel = 20 * np.log10(np.maximum(spectrum / ref, 1e-14))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(f_ghz, db_rel)
    ax.set(xlim=(0, 20), ylim=(-120, 5), xlabel="Frequency (GHz)",
           ylabel="Relative FFT bin magnitude (dBc / bin)",
           title="Transient-window FFT, not phase-noise PSD")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURES / "transient_fft_not_phase_noise.png", dpi=150)
    plt.show()
    print("FFT record: 100–120 ns; 10 ps interpolated spacing; Hann window; bin spacing",
          round(float(f_ghz[1] - f_ghz[0]), 4), "GHz")


## Optional full-resolution 27-case raw sweep (large)

Turn on `RUN_FULL_RAW_GRID` only when you need **every raw time point of all 27 SPICE cases**. This executes the same netlists without `--web` and may produce large CSV files and long runtimes. The default L3 includes all 27 summary/envelope/late-waveform records **plus one complete raw detailed case**.

In [ ]:
if RUN_FULL_RAW_GRID:
    full_raw_seconds = execute(
        "full_raw_27",
        ["--l", "0.8,1,1.2", "--c", "680,810,970", "--rs", "2.2,3.3,18"],
        WORK / "full-raw-27")
    print("Full raw 27-case sweep finished in", round(full_raw_seconds, 1), "s")
else:
    print("Full raw 27-case sweep not requested.")


In [ ]:
provenance = {
    "experiment": "VCO-01",
    "source_ref": SOURCE_REF,
    "zip_sha256": hashlib.sha256(PACKAGE_ZIP.read_bytes()).hexdigest(),
    "engine": "eecircuit-engine=1.9.0; ngspice 45.2 WASM",
    "model": "SKY130 NMOS TT 27C; source model commit f62031a1be9aefe902d6d54cddd6f59b57627436",
    "node_version": version,
    "python_version": platform.python_version(),
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "grid_cases": 27 if RUN_CANONICAL_GRID else None,
    "detail_case": {
        "l_nH": L_NH, "c_fF": C_FF, "rs_ohm": RS_OHM,
        "w_um": PAIR_W_UM, "length_um": PAIR_L_UM,
        "bias_V": TAIL_BIAS_V, "kick_uA": START_KICK_UA
    } if RUN_DETAILED_CASE else None,
    "limitations": [
        "No PSS/PNoise; FFT is not phase-noise PSD",
        "Fixed capacitance operating point",
        "Steady/decay judgment depends on time window",
        "27-case grid and one full raw waveform by default"
    ],
}
(WORK / "provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
archive = WORK / "VCO-01_results.zip"
include = [WORK / "provenance.json", PACKAGE_ZIP, WORK / "grid_27.log",
           WORK / "detail.log"] + sorted(FIGURES.glob("*.png"))
for directory in [WORK / "L2-export", WORK / "web-grid", WORK / "detailed-case"]:
    if directory.exists():
        include.extend(p for p in directory.rglob("*") if p.is_file())
# Do not duplicate the original source ZIP inside the results archive; source is version-pinned online.
include = [p for p in include if p.exists() and p != PACKAGE_ZIP]
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in include:
        z.write(file, arcname=str(file.relative_to(WORK)))
print("Results archive:", archive, archive.stat().st_size, "bytes")
print("Publish only after a successful run and independent L4 Copy & Edit smoke test.")
